# 11 - Dummy Variables (Variables Ficticias)

Los modelos de regresion trabajan con **numeros**. Pero muchas veces tenemos variables **categoricas** (texto).

Para usarlas en regresion, necesitamos convertirlas a numeros. La forma estandar es crear **dummy variables** (variables binarias 0/1).

In [35]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import statsmodels.api as sm

df = pd.read_csv("../data/housing.csv")

print(df[["price", "square_meters", "property_type"]].head(10))
print(f"\nDistribucion de tipos:")
print(df["property_type"].value_counts())

    price  square_meters  property_type
0  192700             51  Single Family
1  132300             52      Apartment
2  163200             56      Townhouse
3  210100             57  Single Family
4  109500             58      Apartment
5   86000             62      Townhouse
6  189700             63  Single Family
7  161000             64  Single Family
8  113300             64      Apartment
9  193800             66      Apartment

Distribucion de tipos:
property_type
Apartment        53
Single Family    39
Townhouse         8
Name: count, dtype: int64


## 1. El problema: no puedes meter texto en una regresion

Si intentas usar `property_type` directamente, no funciona.

In [36]:
# Esto NO funciona
try:
    lr = LinearRegression()
    lr.fit(df[["square_meters", "property_type"]], df["price"])
except ValueError as e:
    print(f"Error: {e}")
    print("\n=> No puedes meter texto directamente en un modelo numerico")

Error: could not convert string to float: 'Single Family'

=> No puedes meter texto directamente en un modelo numerico


## 2. One Hot Encoding: convertir categorias a columnas binarias

La solucion es crear **una columna por cada categoria**, con valores 0 o 1.

En la comunidad de machine learning esto se llama **one hot encoding**.

In [37]:
# One hot encoding con sklearn
encoder = OneHotEncoder(sparse_output=False)
dummies = encoder.fit_transform(df[["property_type"]])
categories = encoder.get_feature_names_out()

print("One Hot Encoding (P columnas):")
print(pd.DataFrame(dummies, columns=categories).head(10).to_string())
print(f"\nColumnas creadas: {list(categories)}")
print(f"3 categorias -> 3 columnas binarias")

One Hot Encoding (P columnas):
   property_type_Apartment  property_type_Single Family  property_type_Townhouse
0                      0.0                          1.0                      0.0
1                      1.0                          0.0                      0.0
2                      0.0                          0.0                      1.0
3                      0.0                          1.0                      0.0
4                      1.0                          0.0                      0.0
5                      0.0                          0.0                      1.0
6                      0.0                          1.0                      0.0
7                      0.0                          1.0                      0.0
8                      1.0                          0.0                      0.0
9                      1.0                          0.0                      0.0

Columnas creadas: ['property_type_Apartment', 'property_type_Single Family', 

## 3. El problema de multicolinealidad: por que usar P-1 columnas

Con 3 columnas de one hot encoding, la tercera es **redundante** — si sabes que no es Apartment (0) ni Single Family (0), **tiene que ser** Townhouse (1).

En regresion, esto causa **multicolinealidad perfecta** porque una columna es combinacion lineal de las otras.

In [38]:
# Demostrar la redundancia
dummies_df = pd.DataFrame(dummies, columns=categories)
print("Observa: la suma de cada fila SIEMPRE es 1")
print(dummies_df.head(10).to_string())
print(f"\nSuma por fila: {dummies_df.sum(axis=1).unique()}")
print("\n=> Si Apartment=0 y Single Family=0, SABEMOS que Townhouse=1")
print("   La tercera columna no aporta informacion nueva")
print("   Y rompe la regresion porque es combinacion lineal de las otras")

Observa: la suma de cada fila SIEMPRE es 1
   property_type_Apartment  property_type_Single Family  property_type_Townhouse
0                      0.0                          1.0                      0.0
1                      1.0                          0.0                      0.0
2                      0.0                          0.0                      1.0
3                      0.0                          1.0                      0.0
4                      1.0                          0.0                      0.0
5                      0.0                          0.0                      1.0
6                      0.0                          1.0                      0.0
7                      0.0                          1.0                      0.0
8                      1.0                          0.0                      0.0
9                      1.0                          0.0                      0.0

Suma por fila: [1.]

=> Si Apartment=0 y Single Family=0, SABEMOS

In [39]:
# Solucion: drop="first" -> P-1 columnas
encoder_dropped = OneHotEncoder(sparse_output=False, drop="first")
dummies_dropped = encoder_dropped.fit_transform(df[["property_type"]])
categories_dropped = encoder_dropped.get_feature_names_out()

print("Con drop='first' (P-1 columnas):")
print(pd.DataFrame(dummies_dropped, columns=categories_dropped).head(10).to_string())
print(f"\nColumnas: {list(categories_dropped)}")
print(f"'Apartment' es la REFERENCIA (cuando ambas columnas son 0)")

Con drop='first' (P-1 columnas):
   property_type_Single Family  property_type_Townhouse
0                          1.0                      0.0
1                          0.0                      0.0
2                          0.0                      1.0
3                          1.0                      0.0
4                          0.0                      0.0
5                          0.0                      1.0
6                          1.0                      0.0
7                          1.0                      0.0
8                          0.0                      0.0
9                          0.0                      0.0

Columnas: ['property_type_Single Family', 'property_type_Townhouse']
'Apartment' es la REFERENCIA (cuando ambas columnas son 0)


In [41]:
# Preparar datos con ColumnTransformer + Pipeline
predictors_num = ["square_meters", "bathrooms", "build_quality"]
predictors_cat = ["property_type"]
all_predictors = predictors_num + predictors_cat

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", predictors_num),
        ("cat", OneHotEncoder(drop="first"), predictors_cat),
    ]
)

pipe = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

X = df[all_predictors]
y = df["price"].to_numpy()

pipe.fit(X, y)

# Ver las columnas resultantes
feature_names = predictors_num + list(
    pipe.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out()
)
print("Columnas del modelo:")
print(feature_names)
print(f"\nForma transformada: {pipe.named_steps['preprocessor'].transform(X).shape}")

Columnas del modelo:
['square_meters', 'bathrooms', 'build_quality', 'property_type_Single Family', 'property_type_Townhouse']

Forma transformada: (100, 5)


In [42]:
# Ver coeficientes
lr = pipe.named_steps["model"]

print("Coeficientes del modelo:")
print(f"  Intercepto: {lr.intercept_:,.0f}")
for name, coef in zip(feature_names, lr.coef_):
    print(f"  {name}: {coef:,.2f}")

print("\nInterpretacion de los coeficientes categoricos:")
print("  (todo es RELATIVO a Apartment, que es la referencia)")
for name, coef in zip(feature_names, lr.coef_):
    if "property_type" in name:
        tipo = name.replace("property_type_", "")
        mas_menos = "mas" if coef > 0 else "menos"
        print(
            f"  {tipo} vale ${abs(coef):,.0f} {mas_menos} que un Apartment (mismas caracteristicas)"
        )

Coeficientes del modelo:
  Intercepto: 2,515
  square_meters: 1,265.35
  bathrooms: 17,704.97
  build_quality: 7,380.09
  property_type_Single Family: 45,979.60
  property_type_Townhouse: -11,473.47

Interpretacion de los coeficientes categoricos:
  (todo es RELATIVO a Apartment, que es la referencia)
  Single Family vale $45,980 mas que un Apartment (mismas caracteristicas)
  Townhouse vale $11,473 menos que un Apartment (mismas caracteristicas)


## 5. Comparar: con dummies vs sin variable categorica

In [44]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", predictors_num),
    ]
)

# Modelo SIN property_type
pipe_sin = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)
scores_sin = cross_val_score(pipe_sin, df[predictors_num], y, cv=5, scoring="r2")

# Modelo CON property_type (dummies)
scores_con = cross_val_score(pipe, df[all_predictors], y, cv=5, scoring="r2")

print(f"R² sin property_type: {scores_sin.mean():.4f}")
print(f"R² con property_type: {scores_con.mean():.4f}")
print(
    f"\n=> Agregar la variable categorica {'mejora' if scores_con.mean() > scores_sin.mean() else 'no mejora'} el modelo"
)

R² sin property_type: 0.1498
R² con property_type: 0.3240

=> Agregar la variable categorica mejora el modelo


## 6. Que pasa si usas P columnas en lugar de P-1?

Veamos el efecto de la multicolinealidad.

In [50]:
# Con statsmodels podemos ver el problema
# P-1 columnas (correcto)
X_transformed = pipe.named_steps["preprocessor"].transform(df[all_predictors])
X_correct_const = sm.add_constant(X_transformed)
model_correct = sm.OLS(y, X_correct_const).fit()

print("=== Con P-1 columnas (correcto) ===")
print(model_correct.summary().tables[1])

# P columnas (problematico)
preprocessor_full = ColumnTransformer(
    transformers=[
        ("num", "passthrough", predictors_num),
        ("cat", OneHotEncoder(drop=None), predictors_cat),
    ]
)
X_full = preprocessor_full.fit_transform(df[all_predictors])
X_full_const = sm.add_constant(X_full)
model_all = sm.OLS(y, X_full_const).fit()


print("\n=== Con P columnas (multicolinealidad) ===")
print(model_all.summary().tables[1])
print("\n=> Con P columnas, statsmodels detecta la redundancia.")
print("   Los errores estandar pueden inflarse y los coeficientes ser inestables.")

=== Con P-1 columnas (correcto) ===
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       2515.2922   2.08e+04      0.121      0.904   -3.88e+04    4.38e+04
x1          1265.3521    183.044      6.913      0.000     901.914    1628.791
x2           1.77e+04   7815.544      2.265      0.026    2187.023    3.32e+04
x3          7380.0944   2961.412      2.492      0.014    1500.141    1.33e+04
x4          4.598e+04   9200.790      4.997      0.000    2.77e+04    6.42e+04
x5         -1.147e+04   1.66e+04     -0.690      0.492   -4.45e+04    2.15e+04

=== Con P columnas (multicolinealidad) ===
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.051e+04   1.52e+04      0.691      0.492   -1.97e+04    4.07e+04
x1          1265.3521    183.044      6.913      0.

## 7. Como interpretar los coeficientes

Los coeficientes de las dummy variables se interpretan **relativos a la categoria de referencia** (la que se elimino).

In [49]:
# Prediccion concreta para entender la interpretacion
# Pasamos DataFrames con las columnas originales — el pipeline transforma automaticamente
casa_base = {"square_meters": 60, "bathrooms": 2, "build_quality": 5}

apartment = pd.DataFrame({**casa_base, "property_type": ["Apartment"]})
single_family = pd.DataFrame({**casa_base, "property_type": ["Single Family"]})
townhouse = pd.DataFrame({**casa_base, "property_type": ["Townhouse"]})

pred_apt = pipe.predict(apartment)[0]
pred_sf = pipe.predict(single_family)[0]
pred_th = pipe.predict(townhouse)[0]

print("Misma casa (60m², 2 banos, calidad 5), diferente tipo:")
print(f"  Apartment:     ${pred_apt:,.0f} (referencia)")
print(f"  Single Family: ${pred_sf:,.0f}")
print(f"  Townhouse:     ${pred_th:,.0f}")

print(f"\nDiferencia Single Family vs Apartment: ${pred_sf - pred_apt:,.0f}")
print(f"Diferencia Townhouse vs Apartment:     ${pred_th - pred_apt:,.0f}")
print("\n=> Estas diferencias son exactamente los coeficientes de las dummy variables")

Misma casa (60m², 2 banos, calidad 5), diferente tipo:
  Apartment:     $150,747 (referencia)
  Single Family: $196,726
  Townhouse:     $139,273

Diferencia Single Family vs Apartment: $45,980
Diferencia Townhouse vs Apartment:     $-11,473

=> Estas diferencias son exactamente los coeficientes de las dummy variables


## 8. Resumen

| Concepto | Descripcion |
|----------|-------------|
| **Dummy variable** | Columna binaria (0/1) que representa una categoria |
| **One hot encoding** | Crear P columnas para P categorias. Usado en ML (arboles, KNN) |
| **Drop first (P-1)** | Crear P-1 columnas, la categoria eliminada es la **referencia**. Usado en regresion |
| **Por que P-1?** | Con intercepto + P columnas hay multicolinealidad perfecta (la suma siempre es 1) |
| **Interpretacion** | Cada coeficiente dummy = diferencia respecto a la categoria de referencia |
| **`OneHotEncoder`** | `drop=None` para one hot, `drop="first"` para regresion |
| **`ColumnTransformer`** | Define transformaciones por tipo de columna (numerica vs categorica) |
| **`Pipeline`** | Encadena preprocesamiento + modelo. Forma profesional y segura |